In [1]:
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any

# Sklearn for scaling + CV
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# SciPy for acquisition functions and optimisation
from scipy.stats import norm
from scipy.optimize import minimize

# PyTorch for deep ensemble surrogate
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# 0. Global settings (Week 7 defaults; will be overridden by tuning)
# ============================================================

RANDOM_STATE = 123
DEVICE = torch.device("cpu")  # change to "cuda" if you have GPU

# Hyperband-style tuning controls
TUNE_N_TRIALS = 24          # random configs to sample
TUNE_KFOLDS = 3             # CV folds
TUNE_STAGE_EPOCHS = [200, 600, 1200]   # successive halving budgets
TUNE_KEEP_FRAC = 0.33       # keep top ~1/3 each stage

# Final optimisation (after tuning)
FINAL_N_CANDIDATES = 60_000  # slightly higher than Week 6
FINAL_N_RESTARTS = 12
FINAL_XI_DEFAULT = 0.01

# ============================================================
# 1. Data (as provided)
# ============================================================

X_raw = np.array([
 [0.60499445, 0.29221502, 0.90845275, 0.35550624, 0.20166872, 0.57533801, 0.31031095, 0.73428138],
 [0.17800696, 0.56622265, 0.99486184, 0.21032501, 0.32015266, 0.70790879, 0.63538449, 0.10713163],
 [0.00907698, 0.81162615, 0.52052036, 0.07568668, 0.26511183, 0.09165169, 0.59241515, 0.36732026],
 [0.50602816, 0.65373012, 0.36341078, 0.17798105, 0.0937283, 0.19742533, 0.7558269, 0.29247234],
 [0.35990926, 0.24907568, 0.49599717, 0.70921498, 0.11498719, 0.28920692, 0.55729515, 0.59388173],
 [0.77881834, 0.0034195, 0.33798313, 0.51952778, 0.82090699, 0.53724669, 0.5513471, 0.66003209],
 [0.90864932, 0.0622497, 0.23825955, 0.76660355, 0.13233596, 0.99024381, 0.68806782, 0.74249594],
 [0.58637144, 0.88073573, 0.74502075, 0.54603485, 0.00964888, 0.74899176, 0.23090707, 0.09791562],
 [0.76113733, 0.85467239, 0.38212433, 0.33735198, 0.68970832, 0.30985305, 0.63137968, 0.04195607],
 [0.9849332, 0.69950626, 0.9988855, 0.18014846, 0.58014315, 0.23108719, 0.49082694, 0.31368272],
 [0.11207131, 0.43773566, 0.59659878, 0.59277563, 0.22698177, 0.41010452, 0.92123758, 0.67475276],
 [0.79188751, 0.57619134, 0.69452836, 0.28342378, 0.13675546, 0.27916186, 0.84276726, 0.62532792],
 [0.1435503, 0.93741452, 0.23232482, 0.00904349, 0.41457893, 0.40932517, 0.55377852, 0.2058408],
 [0.76991655, 0.45875909, 0.55900044, 0.69460444, 0.50319902, 0.72834638, 0.78425353, 0.66313109],
 [0.05644741, 0.06595555, 0.02292868, 0.03878647, 0.40393544, 0.80105533, 0.48830701, 0.89308498],
 [0.86243745, 0.48273382, 0.2818694, 0.54410223, 0.88749026, 0.38265469, 0.60190199, 0.47646169],
 [0.3515119, 0.59006494, 0.9094363, 0.67840835, 0.21282566, 0.08846038, 0.410153, 0.19572429],
 [0.73590364, 0.03461189, 0.72803027, 0.14742652, 0.29574314, 0.44511731, 0.97517969, 0.37433978],
 [0.68029397, 0.25510465, 0.86218799, 0.13439582, 0.3263292, 0.28790687, 0.43501048, 0.36420013],
 [0.04432925, 0.01358149, 0.25819824, 0.57764416, 0.05127992, 0.15856307, 0.59103012, 0.07795293],
 [0.77834548, 0.75114565, 0.31414221, 0.90298577, 0.33538166, 0.38632267, 0.74897249, 0.9887551],
 [0.89888711, 0.5236417, 0.87678325, 0.21869645, 0.90026089, 0.28276624, 0.91107791, 0.47239822],
 [0.14512029, 0.11932754, 0.42088822, 0.38760861, 0.15542283, 0.87517163, 0.51055967, 0.72861058],
 [0.33895442, 0.56693202, 0.3767511, 0.09891573, 0.65945169, 0.24554809, 0.76248278, 0.73215347],
 [0.17615002, 0.29396143, 0.97567997, 0.79393631, 0.92340076, 0.03084229, 0.80325452, 0.59589758],
 [0.02894663, 0.02827906, 0.48137155, 0.6131746, 0.67266045, 0.02211341, 0.6014833, 0.52488505],
 [0.19263987, 0.63067728, 0.41679584, 0.49052929, 0.79608602, 0.65456706, 0.27624119, 0.29551759],
 [0.94318502, 0.21885062, 0.72118408, 0.42459707, 0.986902, 0.53518298, 0.71474318, 0.96009372],
 [0.5327214, 0.8336926, 0.071399, 0.11681148, 0.73069311, 0.93737559, 0.86650798, 0.127902],
 [0.44709584, 0.84395253, 0.72954612, 0.63915138, 0.40928714, 0.13264569, 0.03590888, 0.44683847],
 [0.38222497, 0.55713584, 0.85310163, 0.33379569, 0.26572127, 0.48087292, 0.23764706, 0.76863196],
 [0.53281953, 0.86230848, 0.53826712, 0.04944293, 0.71970119, 0.9067059, 0.10823094, 0.52534791],
 [0.39486519, 0.33180167, 0.7407543, 0.69786172, 0.73740444, 0.78377681, 0.25449546, 0.87114551],
 [0.98594539, 0.87305363, 0.07039262, 0.05358729, 0.73415296, 0.52025852, 0.81104004, 0.10336036],
 [0.96457339, 0.97397979, 0.66375335, 0.66221599, 0.67312167, 0.90523762, 0.45887462, 0.5609175],
 [0.47207071, 0.16820264, 0.08642757, 0.45265551, 0.48061922, 0.62243949, 0.92897446, 0.11253627],
 [0.85600695, 0.6388937, 0.32619202, 0.66850311, 0.24029837, 0.21029889, 0.16754636, 0.96358986],
 [0.81003174, 0.63504604, 0.26954758, 0.86960534, 0.66192159, 0.25225873, 0.76567003, 0.89054867],
 [0.79625252, 0.00703653, 0.35569738, 0.48756605, 0.74051962, 0.7066501, 0.99291449, 0.38173437],
 [0.48124533, 0.10246072, 0.21948594, 0.67732237, 0.24750919, 0.24434086, 0.16382453, 0.71596164],
 [1.085945, 1.073979, 1.098885, 1.002985, 1.086901, 1.090243, 1.092914, 1.092915],
 [1.00000e-06, 2.05513e-01, 1.00000e-06, 8.42570e-02, 8.08305e-01, 3.55768e-01, 1.00000e-06, 4.52510e-01],
 [0.025554, 0.341676, 0.022591, 0.351417, 0.540332, 0.56768 , 0.214489, 0.595807],
 [0.157856, 0.176691, 0.024491, 0.477665, 0.906327, 0.715038, 0.022394, 0.250892],
 [0.026581, 0.517116, 0.045489, 0.164555, 0.986197, 0.391204, 0.022672, 0.723886],
 [0.      , 0.171985, 0.      , 0.282222, 1.      , 1.      , 0.      , 1.      ]
])

y_raw = np.array([
    7.3987211, 7.00522736, 8.45948162, 8.28400781, 8.60611679,
    8.54174792, 7.32743458, 7.29987205, 7.95787474, 5.59219339,
    7.85454099, 6.79198578, 8.97655402, 7.3790829,  9.598482,
    8.15998319, 7.13162397, 6.76796253, 7.43374407, 9.01307515,
    7.31089382, 5.84106731, 9.14163949, 8.81755844, 6.45194313,
    8.83074505, 9.34427428, 6.88784639, 8.04221254, 7.69236805,
    7.92375877, 8.42175924, 8.2780624,  7.11345716, 6.40258841,
    8.47293632, 7.97768459, 7.46087219, 7.43659353, 9.18300525,
    1.6498596481450019,	9.8188854584285, 9.8382810715811, 9.7246626786321, 9.7392105237459,
	9.545334002491
])

assert X_raw.shape[0] == y_raw.shape[0], "X and y sizes must match"

# ============================================================
# 2. PyTorch Deep Ensemble Surrogate (tunable)
# ============================================================

class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(128, 128, 64), dropout=0.0):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(p=dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def build_ensemble_torch(n_members: int, input_dim: int, hidden_dims, dropout) -> List[MLPRegressorTorch]:
    torch.manual_seed(RANDOM_STATE)
    ensemble = []
    for _ in range(n_members):
        model = MLPRegressorTorch(input_dim, hidden_dims=hidden_dims, dropout=dropout).to(DEVICE)
        ensemble.append(model)
    return ensemble


def fit_ensemble_torch(
    ensemble: List[MLPRegressorTorch],
    X_scaled: np.ndarray,
    y_scaled: np.ndarray,
    n_epochs: int,
    batch_size: int,
    lr: float,
    weight_decay: float,
) -> None:
    rng = np.random.RandomState(RANDOM_STATE + 1)
    X = torch.from_numpy(X_scaled.astype(np.float32)).to(DEVICE)
    y = torch.from_numpy(y_scaled.astype(np.float32)).view(-1, 1).to(DEVICE)

    n_samples = X.shape[0]

    for model in ensemble:
        idx = rng.randint(0, n_samples, size=n_samples)
        X_boot = X[idx]
        y_boot = y[idx]

        dataset = TensorDataset(X_boot, y_boot)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.MSELoss()

        model.train()
        for _ in range(n_epochs):
            for xb, yb in loader:
                optimizer.zero_grad()
                preds = model(xb)
                loss = criterion(preds, yb)
                loss.backward()
                optimizer.step()


def ensemble_predict_torch(ensemble: List[MLPRegressorTorch], X_scaled: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X_t = torch.from_numpy(X_scaled.astype(np.float32)).to(DEVICE)
    preds_list = []
    for model in ensemble:
        model.eval()
        with torch.no_grad():
            preds = model(X_t).cpu().numpy().ravel()
        preds_list.append(preds)
    preds = np.stack(preds_list, axis=0)  # (n_members, N)
    mu = preds.mean(axis=0)
    sigma = preds.std(axis=0, ddof=1) + 1e-9
    return mu, sigma


# ============================================================
# 3. Acquisition functions
# ============================================================

def probability_of_improvement(mu: np.ndarray, sigma: np.ndarray, best_y: float, xi: float = 0.01) -> np.ndarray:
    sigma_safe = np.maximum(sigma, 1e-12)
    z = (mu - best_y - xi) / sigma_safe
    return norm.cdf(z)


def expected_improvement(mu: np.ndarray, sigma: np.ndarray, best_y: float, xi: float = 0.01) -> np.ndarray:
    sigma_safe = np.maximum(sigma, 1e-12)
    z = (mu - best_y - xi) / sigma_safe
    ei = (mu - best_y - xi) * norm.cdf(z) + sigma_safe * norm.pdf(z)
    ei[sigma_safe < 1e-12] = 0.0
    return ei


# ============================================================
# 4. EI and gradient for a single point (PyTorch autograd)
# ============================================================

def ei_and_grad_single(
    x_raw_1d: np.ndarray,
    ensemble: List[MLPRegressorTorch],
    x_scaler: MinMaxScaler,
    y_scaler: StandardScaler,
    best_y: float,
    xi: float,
) -> Tuple[float, np.ndarray]:
    x_raw = x_raw_1d.reshape(1, -1)
    x_scaled = x_scaler.transform(x_raw)
    x_t = torch.tensor(x_scaled.astype(np.float32), requires_grad=True, device=DEVICE)

    preds = []
    for model in ensemble:
        model.eval()
        preds.append(model(x_t))

    Y = torch.stack(preds, dim=0).squeeze(-1).squeeze(-1)
    mu_scaled = Y.mean()
    sigma_scaled = Y.std(unbiased=True) + 1e-9

    scale_y = float(y_scaler.scale_[0])
    mean_y = float(y_scaler.mean_[0])
    mu_y = mu_scaled * scale_y + mean_y
    sigma_y = sigma_scaled * scale_y
    sigma_y_safe = torch.clamp(sigma_y, min=1e-8)

    normal = torch.distributions.Normal(0.0, 1.0)
    z = (mu_y - best_y - xi) / sigma_y_safe
    cdf = normal.cdf(z)
    pdf = torch.exp(normal.log_prob(z))

    ei = (mu_y - best_y - xi) * cdf + sigma_y_safe * pdf

    ei.backward()
    grad_x_scaled = x_t.grad.detach().cpu().numpy().reshape(-1)
    grad_x_raw = grad_x_scaled * x_scaler.scale_.astype(np.float32)
    return float(ei.detach().cpu().item()), grad_x_raw


# ============================================================
# 5. Hyperparameter tuning (Random search + successive halving)
# ============================================================

def sample_config(rng: np.random.RandomState) -> Dict[str, Any]:
    hidden_choices = [
        (64, 64),
        (128, 64),
        (128, 128, 64),
        (256, 128, 64),
    ]
    cfg = {
        "hidden_dims": hidden_choices[rng.randint(len(hidden_choices))],
        "dropout": rng.choice([0.0, 0.05, 0.1, 0.2]),
        "lr": 10 ** rng.uniform(-3.3, -2.2),  # ~5e-4 to ~6e-3
        "weight_decay": 10 ** rng.uniform(-6.0, -3.8),  # ~1e-6 to ~1.6e-4
        "batch_size": int(rng.choice([8, 16, 32])),
        "n_ensemble_cv": int(rng.choice([5, 7, 9])),  # smaller for CV speed
        "xi": float(rng.choice([0.0, 0.005, 0.01, 0.02])),
    }
    return cfg


def cv_mse_for_config(
    cfg: Dict[str, Any],
    X_scaled: np.ndarray,
    y_scaled: np.ndarray,
    input_dim: int,
    n_epochs: int,
    n_folds: int,
) -> float:
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    fold_mse = []

    for tr_idx, va_idx in kf.split(X_scaled):
        X_tr, X_va = X_scaled[tr_idx], X_scaled[va_idx]
        y_tr, y_va = y_scaled[tr_idx], y_scaled[va_idx]

        ensemble = build_ensemble_torch(
            n_members=cfg["n_ensemble_cv"],
            input_dim=input_dim,
            hidden_dims=cfg["hidden_dims"],
            dropout=cfg["dropout"],
        )

        fit_ensemble_torch(
            ensemble=ensemble,
            X_scaled=X_tr,
            y_scaled=y_tr,
            n_epochs=n_epochs,
            batch_size=cfg["batch_size"],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
        )

        mu_va, _ = ensemble_predict_torch(ensemble, X_va)
        mse = mean_squared_error(y_va, mu_va)
        fold_mse.append(mse)

    return float(np.mean(fold_mse))


def tune_hyperparameters(X_raw: np.ndarray, y_raw: np.ndarray) -> Dict[str, Any]:
    rng = np.random.RandomState(RANDOM_STATE + 99)

    x_scaler = MinMaxScaler(feature_range=(0.0, 1.0))
    X_scaled = x_scaler.fit_transform(X_raw)

    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

    input_dim = X_raw.shape[1]

    # Sample initial pool
    configs = [sample_config(rng) for _ in range(TUNE_N_TRIALS)]

    # Successive halving across epoch budgets
    survivors = configs
    history = []

    for stage, budget in enumerate(TUNE_STAGE_EPOCHS, start=1):
        scores = []
        for cfg in survivors:
            mse = cv_mse_for_config(cfg, X_scaled, y_scaled, input_dim, n_epochs=budget, n_folds=TUNE_KFOLDS)
            scores.append(mse)
            history.append((stage, budget, mse, cfg))

        # Keep best fraction
        order = np.argsort(scores)  # lower MSE is better
        k_keep = max(3, int(np.ceil(len(survivors) * TUNE_KEEP_FRAC)))
        survivors = [survivors[i] for i in order[:k_keep]]

    # Best overall from last stage
    best_cfg = survivors[0]

    # Add final-fit ensemble size (use larger than CV for stronger uncertainty)
    best_cfg["n_ensemble_final"] = int(np.clip(best_cfg["n_ensemble_cv"] + 6, 10, 15))
    best_cfg["n_epochs_final"] = int(TUNE_STAGE_EPOCHS[-1])
    return best_cfg


# ============================================================
# 6. BO result dataclass
# ============================================================

@dataclass
class BOResult:
    x_next: np.ndarray
    mu_next: float
    sigma_next: float
    pi_next: float
    ei_next: float
    x_best_observed: np.ndarray
    y_best_observed: float
    tuned_config: Dict[str, Any]
    reasoning: str


# ============================================================
# 7. Propose next query point
# ============================================================

def propose_next_point(
    ensemble: List[MLPRegressorTorch],
    x_scaler: MinMaxScaler,
    y_scaler: StandardScaler,
    X_raw: np.ndarray,
    y_raw: np.ndarray,
    n_candidates: int,
    n_restarts: int,
    xi: float,
    random_seed: int,
) -> BOResult:

    rng = np.random.RandomState(random_seed)

    best_idx = np.argmax(y_raw)
    x_best_obs = X_raw[best_idx]
    y_best_obs = float(y_raw[best_idx])

    dim = X_raw.shape[1]
    X_cand_raw = rng.rand(n_candidates, dim)

    X_cand_scaled = x_scaler.transform(X_cand_raw)
    mu_scaled, sigma_scaled = ensemble_predict_torch(ensemble, X_cand_scaled)

    mu = y_scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
    sigma = sigma_scaled * y_scaler.scale_[0]

    ei = expected_improvement(mu, sigma, y_best_obs, xi=xi)
    pi = probability_of_improvement(mu, sigma, y_best_obs, xi=xi)

    best_cand_idx = np.argmax(ei)
    x_next_raw = X_cand_raw[best_cand_idx]
    mu_next = float(mu[best_cand_idx])
    sigma_next = float(sigma[best_cand_idx])
    pi_next = float(pi[best_cand_idx])
    ei_next = float(ei[best_cand_idx])

    def objective(x: np.ndarray) -> Tuple[float, np.ndarray]:
        ei_val, grad = ei_and_grad_single(
            x_raw_1d=x,
            ensemble=ensemble,
            x_scaler=x_scaler,
            y_scaler=y_scaler,
            best_y=y_best_obs,
            xi=xi,
        )
        return -ei_val, -grad

    bounds = [(0.0, 1.0)] * dim
    top_indices = np.argsort(-ei)[:n_restarts]
    best_f = -ei_next
    best_x = x_next_raw.copy()

    for idx in top_indices:
        x0 = X_cand_raw[idx]
        res = minimize(
            fun=lambda x: objective(x)[0],
            x0=x0,
            method="L-BFGS-B",
            jac=lambda x: objective(x)[1],
            bounds=bounds,
            options={"maxiter": 120},
        )
        if res.success and res.fun < best_f:
            best_f = res.fun
            best_x = res.x

    x_refined_raw = best_x.reshape(1, -1)
    x_refined_scaled = x_scaler.transform(x_refined_raw)

    mu_scaled_ref, sigma_scaled_ref = ensemble_predict_torch(ensemble, x_refined_scaled)
    mu_ref = float(y_scaler.inverse_transform(mu_scaled_ref.reshape(-1, 1))[0, 0])
    sigma_ref = float(sigma_scaled_ref[0] * y_scaler.scale_[0])

    pi_ref = float(probability_of_improvement(np.array([mu_ref]), np.array([sigma_ref]), y_best_obs, xi=xi)[0])
    ei_ref = float(expected_improvement(np.array([mu_ref]), np.array([sigma_ref]), y_best_obs, xi=xi)[0])

    reasoning = "\n".join([
        f"Observed best: y_best = {y_best_obs:.6f} at x_best = {x_best_obs}.",
        f"Candidate search: sampled {n_candidates} points in [0,1]^{dim}.",
        f"Exploration/exploitation: xi = {xi}.",
        f"Random-search best EI: EI={ei_next:.6f} at x≈{x_next_raw}.",
        f"Refinement: L-BFGS-B from top {n_restarts} EI seeds improved to EI={ei_ref:.6f}.",
        f"Final x_next≈{x_refined_raw.ravel()}, μ≈{mu_ref:.6f}, σ≈{sigma_ref:.6f}, PI≈{pi_ref:.3f}.",
        "Because EI increased after refinement, the refined point is recommended as the next query."
    ])

    return x_refined_raw.ravel(), mu_ref, sigma_ref, pi_ref, ei_ref, x_best_obs, y_best_obs, reasoning


# ============================================================
# 8. Main script (Week 7)
# ============================================================

def main():
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)

    print("======================================================")
    print("WEEK 7 FUNCTION 8 — HYPERPARAMETER TUNING + BO (8D)")
    print("======================================================\n")

    print("TUNING surrogate hyperparameters (random search + successive halving / Hyperband-style)...")
    best_cfg = tune_hyperparameters(X_raw, y_raw)
    print("Best tuned hyperparameters:")
    for k, v in best_cfg.items():
        print(f"  {k}: {v}")
    print()

    # ---- Scale inputs and outputs ----
    x_scaler = MinMaxScaler(feature_range=(0.0, 1.0))
    X_scaled = x_scaler.fit_transform(X_raw)

    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

    # ---- Train FINAL ensemble surrogate with tuned config ----
    print("Training FINAL ensemble surrogate with tuned hyperparameters...")
    input_dim = X_raw.shape[1]
    ensemble = build_ensemble_torch(
        n_members=best_cfg["n_ensemble_final"],
        input_dim=input_dim,
        hidden_dims=best_cfg["hidden_dims"],
        dropout=best_cfg["dropout"],
    )
    fit_ensemble_torch(
        ensemble=ensemble,
        X_scaled=X_scaled,
        y_scaled=y_scaled,
        n_epochs=best_cfg["n_epochs_final"],
        batch_size=best_cfg["batch_size"],
        lr=best_cfg["lr"],
        weight_decay=best_cfg["weight_decay"],
    )
    print("Training complete.\n")

    # ---- Propose next query point ----
    x_next, mu_next, sigma_next, pi_next, ei_next, x_best, y_best, reasoning = propose_next_point(
        ensemble=ensemble,
        x_scaler=x_scaler,
        y_scaler=y_scaler,
        X_raw=X_raw,
        y_raw=y_raw,
        n_candidates=FINAL_N_CANDIDATES,
        n_restarts=FINAL_N_RESTARTS,
        xi=best_cfg["xi"],
        random_seed=RANDOM_STATE + 5,
    )

    # ---- Report ----
    print("=== CURRENT BEST OBSERVED ===")
    print(f"x_best = {x_best}")
    print(f"y_best = {y_best:.6f}\n")

    print("=== RECOMMENDED NEXT POINT ===")
    print(f"x_next     = {x_next}")
    print(f"μ(x_next)  = {mu_next:.6f}")
    print(f"σ(x_next)  = {sigma_next:.6f}")
    print(f"PI         = {pi_next:.3f}")
    print(f"EI         = {ei_next:.6f}\n")

    print("=== REASONING ===")
    print(reasoning)
    print("\nDone.")


if __name__ == "__main__":
    main()


WEEK 7 FUNCTION 8 — HYPERPARAMETER TUNING + BO (8D)

TUNING surrogate hyperparameters (random search + successive halving / Hyperband-style)...
Best tuned hyperparameters:
  hidden_dims: (128, 128, 64)
  dropout: 0.0
  lr: 0.0006143928120769549
  weight_decay: 8.84540078385369e-05
  batch_size: 32
  n_ensemble_cv: 9
  xi: 0.0
  n_ensemble_final: 15
  n_epochs_final: 1200

Training FINAL ensemble surrogate with tuned hyperparameters...
Training complete.

=== CURRENT BEST OBSERVED ===
x_best = [0.025554 0.341676 0.022591 0.351417 0.540332 0.56768  0.214489 0.595807]
y_best = 9.838281

=== RECOMMENDED NEXT POINT ===
x_next     = [0.         0.68070992 0.         0.07341883 1.         0.50995614
 0.         0.        ]
μ(x_next)  = 9.740701
σ(x_next)  = 0.232678
PI         = 0.337
EI         = 0.052080

=== REASONING ===
Observed best: y_best = 9.838281 at x_best = [0.025554 0.341676 0.022591 0.351417 0.540332 0.56768  0.214489 0.595807].
Candidate search: sampled 60000 points in [0,1]^8.